In [ ]:
from torch.utils.data import DataLoader
from datasets import load_dataset

wmt_dataset = load_dataset(
    "wmt/wmt14",
    "de-en",
    split="train",
    # streaming=True
)

In [ ]:
train_loader = DataLoader(wmt_dataset, batch_size=16)


In [ ]:
batch = next(iter(train_loader))

In [ ]:
import tiktoken

gpt2_enc = tiktoken.get_encoding("gpt2")
enc = tiktoken.Encoding(
    name="mygpt2",
    pat_str=gpt2_enc._pat_str,
    mergeable_ranks=gpt2_enc._mergeable_ranks,
    special_tokens={
        "<BOS>": 50256,
        "<EOS>": 50257,
        "<PAD>": 50258
    }
)

In [ ]:
for de in batch['translation']['de']:
    block_size = 512
    x = enc.encode(de) + [50257]
    x = x + (block_size - len(x)) * [50258]
    print(enc.decode(x))

In [1]:
BOS_IDX = 50256
EOS_IDX = 50257
PAD_IDX = 50258

In [ ]:
max_length = 256

def seq_to_tok(x):
    x = enc.encode(x)
    if len(x) > max_length - 1:
        x = x[:max_length - 1]
    return x


In [ ]:
wmt_dataset['translation']

In [ ]:

        
# train_loader = DataLoader(wmt_dataset, batch_size=16)

ds_dict = {
    'de': [],
    'en': []
}

for p in wmt_dataset['translation']:
    s = enc.encode(p['de'])
    d = enc.encode(p['en'])
    
    ds_dict['de'].append(s)
    ds_dict['en'].append(d)


In [ ]:
from datasets import Dataset
ds = Dataset.from_dict(ds_dict)
ds

In [ ]:
print(len(ds['de']))
print(len(ds['en']))

In [ ]:
ds.save_to_disk('./my_wmt14_tokens')

In [3]:
from datasets import load_from_disk
ds = load_from_disk('./my_wmt14_tokens')
len(ds)

4508785

In [16]:
from datasets import load_from_disk
import torch
class DataLoaderLite:
    def __init__(self, B, T):
        self.B = B
        self.T = T
        
        self.ds = load_from_disk('./my_wmt14_tokens')
        self.length = len(self.ds)
        
        self.curr_pos = 0
        
        print(f"load tokens {self.length * 2 * T}")
        print(f"1 epoch = {self.length // B} batches")
        
    def next_batch(self):
        B, T = self.B, self.T
        
        if self.curr_pos > self.length - B:
            self.curr_pos = 0
        
        src_idx = self.ds['de'][self.curr_pos:self.curr_pos + B]
        idx = self.ds['en'][self.curr_pos:self.curr_pos + B]
           
        for i in range(B):
            src_idx[i] += [EOS_IDX]
            if len(src_idx[i]) < T:
                src_idx[i] += [PAD_IDX] * (T - len(src_idx[i]))
                
            idx[i] += [EOS_IDX]
            if len(idx[i]) < T:
                idx[i] += [PAD_IDX] * (T - len(idx[i]))
            idx[i] = [BOS_IDX] + idx[i]
            
        src_idx = torch.tensor(src_idx).view(B, T)
        idx = torch.tensor(idx).view(B, T + 1)
        
        return src_idx, idx[:,:-1], idx[:,1:]
    
ds = DataLoaderLite(16, 256)

load tokens 2308497920
1 epoch = 281799 batches


In [17]:
ds.next_batch()

(tensor([[   54,   798,   263,  ..., 50258, 50258, 50258],
         [   40,   354,  1931,  ..., 50258, 50258, 50258],
         [   54,   494, 48931,  ..., 50258, 50258, 50258],
         ...,
         [49562,    84,  1736,  ..., 50258, 50258, 50258],
         [   42, 48863,   429,  ..., 50258, 50258, 50258],
         [ 5308,   500,  1305,  ..., 50258, 50258, 50258]]),
 tensor([[50256,  4965, 24098,  ..., 50258, 50258, 50258],
         [50256,    40, 13627,  ..., 50258, 50258, 50258],
         [50256,  7003,    11,  ..., 50258, 50258, 50258],
         ...,
         [50256, 18454,   321,  ..., 50258, 50258, 50258],
         [50256,    40,   561,  ..., 50258, 50258, 50258],
         [50256,  3666,  1808,  ..., 50258, 50258, 50258]]),
 tensor([[ 4965, 24098,   286,  ..., 50258, 50258, 50258],
         [   40, 13627, 28291,  ..., 50258, 50258, 50258],
         [ 7003,    11,   355,  ..., 50258, 50258, 50258],
         ...,
         [18454,   321,  1992,  ..., 50258, 50258, 50258],
         [